# islide — annotations demo

Annotations are GeoJSON: import a document from Python, hand-draw a
polygon on the canvas, then select, label, recolor, or delete
individual shapes. Everything lands in the same synced
`v.annotations` document — an imported shape, a drawn one, and an
edited one are all the same kind of feature (DESIGN.md §6.3).

> **Note:** the overlay is drawn by the JupyterLab/notebook extension
> (`frontend/`). Without the built extension the cells below still run
> (parsing + the synced `annotations` trait) but draw no overlay, and
> the A/click drawing gestures have no view to drive.

## 1. Open the slide

The slide opens on a background thread; `wait()` blocks until it's ready.


In [ ]:
from islide import SlideViewer

# Test slide (data/ is gitignored): download data/CMU-1.tiff from
# https://openslide.cs.cmu.edu/download/openslide-testdata/Generic-TIFF/CMU-1.tiff
v = SlideViewer("data/CMU-1.tiff")
v.wait()
v


## 2. Import a GeoJSON document

`set_annotations` accepts a parsed dict, a path to a `.geojson` file, or a
`Path`. The input may be a `FeatureCollection`, a single `Feature`, or a
bare geometry; the stored document is the canonical `FeatureCollection`. Coordinates are **level-0 slide pixels**, y-down from the
origin. Each feature's `properties` may carry
`label` (string), `color` (CSS stroke color, default black) and `fill`
(CSS interior color, default transparent).

We place three shapes near the slide center and then point the view at
them.


In [ ]:
DOC = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "id": "nucleus",
            "geometry": {"type": "Point", "coordinates": [23000, 16450]},
            "properties": {"label": "nucleus"},
        },
        {
            "type": "Feature",
            "geometry": {"type": "LineString",
                           "coordinates": [[22600, 16050], [23400, 16850]]},
            "properties": {"color": "red"},
        },
        {
            "type": "Feature",
            "geometry": {"type": "Polygon",
                           "coordinates": [[[22600, 16050], [23400, 16050],
                                              [23400, 16850], [22600, 16850]]]},
            "properties": {"fill": "rgba(0, 120, 255, 0.35)"},
        },
    ],
}
v.set_annotations(DOC)


Center the view on the imported shapes (slide center `(23000, 16450)`
in level-0 px, zoom 8x) so the overlay is visible:


In [ ]:
v.center_on(23000, 16450)
v.set_zoom(8.0)


## 3. The canonical annotation document

`v.annotations` is the synced **canonical document** (DESIGN.md §6.3): a
plain-JSON GeoJSON `FeatureCollection` of `{id, geometry, properties}`
features in level-0 px. The document is normalized: coordinates are finite
floats, polygon rings are stored *open* (no redundant closing position),
features without an `id` get a fresh `aN` one, and each feature's
`properties` is kept whole (`label`/`color`/`fill` are optional).


In [ ]:
v.annotations


## 4. Clear

`clear_annotations()` empties the overlay.


In [ ]:
v.clear_annotations()
v.annotations


## 5. Draw a polygon

A hand-drawn shape lands in the same `annotations` document as an
imported one — identical document model, identical renderer, identical
`read_crop` story.

How to draw:

1. Click somewhere on the canvas (so the view has keyboard focus) and
   press **A** — the cursor becomes a crosshair and the status bar says
   `Polygon: click to add points — A saves, Esc cancels`.
2. **Left-click** each vertex (~4, around the slide center
   `(23000, 16457)`). A drag still pans and the wheel zoom stays live,
   so the draft tracks the view.
3. Press **A** again to save, or **Esc** to cancel the draft.

On save, the view sends the *open* ring (no redundant closing position),
in unclamped level-0 px, as `last_polygon`; Python normalizes it with the
same shared ring helper the GeoJSON importer uses (≥ 3 points, nonzero
area) and appends a `Polygon` feature to the document. Degenerate drafts
(0–2 clicks, collinear points) are discarded with a warning —
`annotations` is untouched.

In [ ]:
# After the A / click... / A sequence:
# `v.last_polygon` holds the last saved ring (open, unclamped level-0 px)

v.last_polygon

# and the appended `Polygon` feature (fresh id, empty properties):

v.annotations


## 6. Read the crop under the drawn polygon

The polygon is just a feature in `annotations` — e.g. read back the tissue
under it (bounding box, padded, at level 0):


In [ ]:
if not v.annotations["features"]:
    raise RuntimeError(
        "no annotations yet — draw one first (A, click the vertices, A)"
    )

ring = v.annotations["features"][-1]["geometry"]["coordinates"][0]
xs = [p[0] for p in ring]
ys = [p[1] for p in ring]
pad = 256
w, h = v.meta["dimensions"]
bbox = (
    max(0, min(xs) - pad),
    max(0, min(ys) - pad),
    min(w, max(xs) + pad),
    min(h, max(ys) + pad),
)
img = v.read_crop(bbox)
img


## 7. Editing

A shape that already exists — imported or drawn — can be selected in the
view and deleted, labeled, or recolored. There is no geometry editing
(no move / resize / vertex edit) and no multi-select; `v.annotations`
stays the single source of truth, and the view's buttons issue the same
commands the Python API below issues.

The document below carries explicit ids (`roi`, `nucleus`), which the
editing API addresses.

In [ ]:
DOC = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "id": "roi",
            "geometry": {
                "type": "Polygon",
                "coordinates": [[[22700, 16150], [23300, 16150],
                                  [23300, 16750], [22700, 16750]]],
            },
            "properties": {},
        },
        {
            "type": "Feature",
            "id": "nucleus",
            "geometry": {"type": "Point", "coordinates": [23000, 16450]},
            "properties": {"label": "nucleus"},
        },
    ],
}
v.set_annotations(DOC)
v.center_on(23000, 16450)
v.set_zoom(8.0)
v

## 8. Select and edit in the view

In the idle mode (not drawing):

1. **Left-click a shape** to select it — it is highlighted with a thick
   accent stroke (a point gets an accent ring). Clicking empty space
   deselects; clicking another shape moves the selection. A drag of ≥ 4
   px is still a pan, and navigation (pan / zoom / minimap) keeps the
   selection on the shape.
2. While a shape is selected, three toolbar buttons are enabled:
   - **label** — an inline text input appears in the toolbar, prefilled
     with the current label. Enter (or blur) commits; an empty string
     clears the label; Esc cancels. Labels render on
     lines/polygons too (at the first vertex), not just points.
   - **color** — two native color pickers (stroke, fill) plus a
     clear-fill checkbox. Closing either picker commits the full
     `(color, fill)` pair, so you can set the stroke only, the fill
     only, both, or clear the fill.
   - **del** — removes the selected shape.
3. The **annotate** toolbar button toggles polygon drawing (**A**
   stays the keyboard alias); entering drawing mode clears the selection.

Every edit lands back in `v.annotations` (the whole set is pushed), and
the status bar reports it (`deleted #<id>` / `edited #<id> (label|color)`).
After **label** / **color** the selection persists, so edits chain.

## 9. The same ops programmatically

The view's del / label / color buttons issue `annotation_edit` commands
over the comm channel; the public API issues the same commands through
the same trait (ids come from `v.annotations`):

```python
v.delete_annotation(feature_id)
v.set_annotation_label(feature_id, "tumor")   # None / "" clears
v.set_annotation_color(feature_id, color="red", fill=None)  # None = default
```

Unknown or stale ids (e.g. after a `set_annotations()` replace) are
no-ops: the document is untouched and the status says `edit ignored:`.
An id that no longer exists because the shape was deleted in the view is
one of these — the command is a harmless replay.

In [ ]:
v.set_annotation_label("nucleus", "tumor")
v.set_annotation_color("roi", color="red", fill="rgba(255,0,0,0.2)")
v.annotations

## 10. Draw one and delete it

Press **A** (or the **annotate** button), left-click the vertices of a
new polygon, press **A** to save. Select it and press **del** in the
view — or do it here (hand-drawn features get fresh `aN` ids, so they
can be told apart from the imported `roi` / `nucleus`):


In [ ]:
# remove the hand-drawn shapes (ids "a0", "a1", ...); imported ids are
# untouched. If you already deleted them in the view, this is a no-op.
drawn = [f["id"] for f in v.annotations["features"] if f["id"][0] == "a"]
for fid in reversed(drawn):
    v.delete_annotation(fid)
v.annotations

## 11. Clear

`clear_annotations()` removes everything — imported and drawn shapes.
There is no undo; the reset paths are `clear_annotations()` or a re-import.


In [ ]:
v.clear_annotations()
v.annotations